<a href="https://colab.research.google.com/github/Amanicka2/flyrank_mlengineer/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Amanicka2/flyrank_mlengineer/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule flags a page for review if it's moderately stale and still gets real traffic. I checked staleness first since it's the signal behind FlyRank's refresh flags: bucketing pages by days since their last update, decline rate actually falls as pages get staler, from 0.582 in the freshest bucket down to 0.039 in the oldest, the opposite of what I expected. Verdict: OPPOSITE. My guess is content teams update pages because they're already declining, so freshly touched pages are catching problems mid fix, while old untouched pages are often just stable and low priority. I checked volume next, the signal behind the quick win flag: bucketing by March impressions, decline rate peaks in the middle quartile and drops at both ends. Verdict: MIXED. Since the most stale pages turned out to be the safest ones, my rule targets the 31 to 90 day staleness bucket instead of the oldest pages, combined with real traffic, since that's where decline rate is still elevated with a solid sample size. Reason codes: moderately_stale_high_volume, moderately_stale_low_volume, not_flagged.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, pandas as pd, numpy as np
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
march = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
feb   = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"
content = f"read_parquet('{REL}/dim_content.parquet')"

page_level = con.sql(f"""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM {march} GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM {feb} GROUP BY content_hash_id
    )
    SELECT
        m.content_hash_id,
        m.impressions_march,
        COALESCE(f.impressions_feb, 0) AS impressions_feb,
        c.content_updated_date,
        DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') AS days_since_update
    FROM march_agg m
    LEFT JOIN feb_agg f ON m.content_hash_id = f.content_hash_id
    LEFT JOIN {content} c ON m.content_hash_id = c.content_hash_id
""").df()

page_level["trend_pct"] = (page_level["impressions_march"] - page_level["impressions_feb"]) / page_level["impressions_feb"].replace(0, pd.NA) * 100
page_level["is_declining_label"] = (page_level["trend_pct"] < -20).fillna(False).astype(int)
print(page_level.shape[0], "pages | decline rate:", round(page_level["is_declining_label"].mean(), 3))

# Signal 1: staleness, behind FlyRank's refresh flags
bins = [-1, 30, 90, 180, np.inf]
labels = ["0-30", "31-90", "91-180", "181+"]
page_level["staleness_bucket"] = pd.cut(page_level["days_since_update"], bins=bins, labels=labels)
sig1 = (page_level.groupby("staleness_bucket", observed=True)["is_declining_label"]
        .agg(n="count", decline_rate="mean").round(3))
print("\nSignal 1: staleness")
print(sig1)

# Signal 2: volume, behind the quick-win flag
page_level["volume_bucket"] = pd.qcut(page_level["impressions_march"], q=4, duplicates="drop")
sig2 = (page_level.groupby("volume_bucket", observed=True)["is_declining_label"]
        .agg(n="count", decline_rate="mean").round(3))
print("\nSignal 2: volume")
print(sig2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

331437 pages | decline rate: 0.114

Signal 1: staleness
                      n  decline_rate
staleness_bucket                     
0-30                975         0.582
31-90             29680         0.304
91-180             3608         0.143
181+               3816         0.039

Signal 2: volume
                        n  decline_rate
volume_bucket                          
(-0.001, 2.0]      170857         0.089
(2.0, 216.0]        77807         0.185
(216.0, 617124.0]   82773         0.099


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

hi_cut = page_level["impressions_march"].quantile(0.75)   # 216-ish, top quartile
lo_cut = page_level["impressions_march"].quantile(0.25)   # ~2, bottom quartile

is_mod_stale = page_level["staleness_bucket"].eq("31-90")

conditions = [
    is_mod_stale & (page_level["impressions_march"] >= hi_cut),
    is_mod_stale & (page_level["impressions_march"] > lo_cut),
]
reason_choices = ["moderately_stale_high_volume", "moderately_stale_moderate_volume"]
action_choices = ["review", "review"]

page_level["reason_code"] = np.select(conditions, reason_choices, default="not_flagged")
page_level["action"] = np.select(conditions, action_choices, default="monitor")
page_level["score"] = np.select(conditions, [page_level["impressions_march"] * 2, page_level["impressions_march"]], default=0)

queue = page_level.sort_values("score", ascending=False).reset_index(drop=True)
print(queue["reason_code"].value_counts())

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("saved", len(queue), "rows")

reason_code
not_flagged                         305741
moderately_stale_high_volume         14868
moderately_stale_moderate_volume     10828
Name: count, dtype: int64
saved 331437 rows


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)[["content_hash_id", "score", "reason_code", "action",
                         "impressions_march", "days_since_update", "is_declining_label"]]
top20

,content_hash_id,score,reason_code,action,impressions_march,days_since_update,is_declining_label
0,content_f43118e089ecc69a,278834.0,moderately_stale_high_volume,review,139417.0,34,0
1,content_9c057b66c30a3abb,167668.0,moderately_stale_high_volume,review,83834.0,34,1
2,content_73aa61dcedebbf30,160248.0,moderately_stale_high_volume,review,80124.0,34,0
3,content_80eb6221de550658,159532.0,moderately_stale_high_volume,review,79766.0,34,0
4,content_ac7b77e81c53d636,159302.0,moderately_stale_high_volume,review,79651.0,34,0
5,content_49267c758cdcb3a8,147536.0,moderately_stale_high_volume,review,73768.0,34,0
6,content_e9f2d0579387d3c3,147006.0,moderately_stale_high_volume,review,73503.0,34,0
7,content_57dcb96896f9a33c,140052.0,moderately_stale_high_volume,review,70026.0,34,0
8,content_b875a2f306635a58,137314.0,moderately_stale_high_volume,review,68657.0,34,0
9,content_f2388a4b87a3b1dc,135292.0,moderately_stale_high_volume,review,67646.0,34,0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

All 20 of my top picks turned out to be pages that are not currently declining, meaning my rule found zero true positives in its own top 20. This is a real limitation, not a bug: sorting by raw volume within the moderately stale window surfaces FlyRank's biggest, most established pages, which tend to be stable rather than at risk. A stronger rule would need a signal that separates "big and stable" from "big and starting to slip", volume alone can't do that. I also noticed all 20 picks share the same days_since_update value, 34, suggesting a batch update event rather than 20 independently timed edits. Leakage check: the score used only staleness_bucket and impressions_march, is_declining_label, trend_pct, and impressions_feb never touched it.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# weak picks: high score but not actually declining, worth a hard look
weak = top20[top20["is_declining_label"] == 0]
print(weak)

# leakage check: confirm the score never touched the label or its inputs
score_inputs = ["staleness_bucket", "impressions_march"]
banned = ["is_declining_label", "trend_pct", "impressions_feb"]
print("score built only from:", score_inputs)
print("none of these touched the score:", banned)

             content_hash_id     score                   reason_code  action  \
0   content_f43118e089ecc69a  278834.0  moderately_stale_high_volume  review   
2   content_73aa61dcedebbf30  160248.0  moderately_stale_high_volume  review   
3   content_80eb6221de550658  159532.0  moderately_stale_high_volume  review   
4   content_ac7b77e81c53d636  159302.0  moderately_stale_high_volume  review   
5   content_49267c758cdcb3a8  147536.0  moderately_stale_high_volume  review   
6   content_e9f2d0579387d3c3  147006.0  moderately_stale_high_volume  review   
7   content_57dcb96896f9a33c  140052.0  moderately_stale_high_volume  review   
8   content_b875a2f306635a58  137314.0  moderately_stale_high_volume  review   
9   content_f2388a4b87a3b1dc  135292.0  moderately_stale_high_volume  review   
10  content_dd5472aea4c7aa91  134554.0  moderately_stale_high_volume  review   
11  content_59c5cc86fe3744bf  127902.0  moderately_stale_high_volume  review   
12  content_cb42a970d503b0f6  121332.0  

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.